# Задание №4. Классификация текстов на основе CNN

**Выполнил:** Асламин

**Группа:** 

**Дата:** 2026

**Ссылка на репозиторий:** https://github.com/.../text-style-classifier

## 1. Введение

**Цель работы:** построить и обучить нейронную сеть для классификации текстов по стилю (новости, научная литература, художественная литература, реклама, техническая документация) с использованием сверточных нейронных сетей (CNN).

**Задачи:**
- Подготовить размеченный корпус текстов
- Выполнить аугментацию и балансировку данных
- Построить CNN модель для классификации
- Обучить модель и оценить её качество
- Сохранить модель и вспомогательные объекты
- Реализовать функцию предсказания стиля текста

## 2. Теоретическая часть

### 2.1. Корпус (Corpus)
Корпус – это структурированная коллекция текстов, используемая для обучения моделей NLP. В задаче классификации стиля необходим **размеченный корпус** – набор текстов, каждому из которых приписана метка класса (стиля).

### 2.2. Токенизация (Tokenization)
Процесс разбиения текста на минимальные единицы – **токены** (слова, знаки препинания). Для токенизации используем `Tokenizer` из Keras, который преобразует текст в последовательность целочисленных индексов.

### 2.3. Эмбеддинги (Embeddings)
Векторные представления слов. Слой **Embedding** преобразует индексы токенов в плотные векторы фиксированной размерности. Эти векторы обучаются вместе с моделью и позволяют улавливать семантические и синтаксические сходства между словами.

### 2.4. Сверточные нейронные сети (CNN) для текста
Сверточные сети изначально разработаны для обработки изображений, но успешно применяются и к текстам. **Одномерная свертка (Conv1D)** скользит по последовательности эмбеддингов, извлекая локальные признаки (например, характерные сочетания слов). Затем слой **GlobalMaxPooling1D** агрегирует самые важные признаки, после чего несколько полносвязных слоёв выполняют классификацию.

### 2.5. Классификация текста
Задача отнесения текста к одной из заранее известных категорий. Выходной слой модели использует активацию **softmax**, который выдаёт вероятности принадлежности каждому классу. Функция потерь – **categorical_crossentropy**.

### 2.6. Аугментация данных и балансировка классов
Для улучшения обобщающей способности модели и борьбы с дисбалансом классов применяют:
- **Аугментацию текста**: случайное удаление или перестановка слов
- **Балансировку**: добавление синтетических примеров для классов с малым количеством образцов

## 3. Описание данных

### 3.1. Источник данных
В данной работе используется размеченный корпус текстов, содержащий тексты пяти стилей:
- **новости** - новостные тексты
- **научная** - научные статьи
- **художественная** - художественная литература
- **реклама** - рекламные объявления
- **техническая** - техническая документация

Файл с данными: `dataset.txt`

In [1]:
# Импорт необходимых библиотек
import random
import numpy as np
import tensorflow as tf
from keras.models import Sequential, load_model
from keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from keras.utils import to_categorical
from collections import Counter
import matplotlib.pyplot as plt
import pickle
import requests
import os

print("Библиотеки успешно импортированы")
print(f"Версия TensorFlow: {tf.__version__}")

Библиотеки успешно импортированы
Версия TensorFlow: 2.20.0


In [ ]:
# Загрузка датасета
def load_dataset(path, num_samples=None):
    texts, labels = [], []
    with open(path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if num_samples and i >= num_samples:
                break
            parts = line.strip().split('\t')
            if len(parts) != 2:
                continue
            text, label = parts
            texts.append(text)
            labels.append(label)
    return texts, labels

# Загрузка данных
texts, labels = load_dataset('./dataset.txt')

print(f"Загружено примеров: {len(texts)}")
print(f"Распределение классов: {Counter(labels)}")

In [ ]:
# Статистика по данным
print("=" * 50)
print("СТАТИСТИКА ДАННЫХ")
print("=" * 50)

# Количество примеров по классам
class_counts = Counter(labels)
print("\nКоличество примеров по классам:")
for label, count in class_counts.items():
    print(f"  {label}: {count}")

# Средняя длина текста
text_lengths = [len(text.split()) for text in texts]
avg_length = np.mean(text_lengths)
max_length = max(text_lengths)
min_length = min(text_lengths)

print(f"\nДлина текста (в словах):")
print(f"  Средняя: {avg_length:.2f}")
print(f"  Максимальная: {max_length}")
print(f"  Минимальная: {min_length}")

print(f"\nВсего классов: {len(class_counts)}")

## 4. Подготовка данных

### 4.1. Аугментация и балансировка классов

Для улучшения качества модели и борьбы с дисбалансом классов применим аугментацию текста и балансировку. Аугментация позволяет увеличить количество примеров в классах с меньшим количеством данных путём случайного удаления или перестановки слов.

In [ ]:
# Аугментация текста: случайное удаление и перестановка слов
def augment_text(text):
    words = text.split()
    if len(words) <= 3:
        return text
    if random.random() < 0.5:
        idx = random.randrange(len(words))
        words.pop(idx)
    if random.random() < 0.5:
        i, j = random.sample(range(len(words)), 2)
        words[i], words[j] = words[j], words[i]
    return ' '.join(words)

# Балансировка классов через аугментацию
def balance_dataset(texts, labels, min_count=100):
    counter = Counter(labels)
    new_texts, new_labels = list(texts), list(labels)
    
    for label, count in counter.items():
        to_augment = [t for t, l in zip(texts, labels) if l == label]
        needed = max(0, min_count - count)
        
        for _ in range(needed):
            orig = random.choice(to_augment)
            new_t = augment_text(orig)
            new_texts.append(new_t)
            new_labels.append(label)
    
    return new_texts, new_labels

# Применяем балансировку (минимум 100 примеров на класс)
random.seed(42)
texts_bal, labels_bal = balance_dataset(texts, labels, min_count=100)

print(f"После балансировки: {len(texts_bal)} примеров")
print(f"Распределение: {Counter(labels_bal)}")

### 4.2. Токенизация и паддинг

Токенизация преобразует текст в последовательность целых чисел. Паддинг дополняет все последовательности до одинаковой длины.

In [ ]:
# Параметры токенизации
VOCAB_SIZE = 10000     # максимальный размер словаря
OOV_TOKEN = '<OOV>'    # токен для неизвестных слов
MAX_LENGTH = 50        # максимальная длина текста (в словах)

# Создание и обучение токенизатора
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(texts_bal)

# Преобразование текстов в последовательности
sequences = tokenizer.texts_to_sequences(texts_bal)

# Паддинг последовательностей
X = pad_sequences(sequences, maxlen=MAX_LENGTH, padding='post', truncating='post')

print(f"Размерность X: {X.shape}")
print(f"Размер словаря: {len(tokenizer.word_index)}")

In [ ]:
# Кодирование меток
label_encoder = LabelEncoder()
y_int = label_encoder.fit_transform(labels_bal)
y = to_categorical(y_int)
num_classes = y.shape[1]

print(f"Число классов: {num_classes}")
print(f"Метки: {label_encoder.classes_}")
print(f"Размерность y: {y.shape}")

### 4.3. Разделение на обучающую и тестовую выборки

Используем стратифицированное разделение (80/20) для сохранения пропорций классов.

In [ ]:
# Разделение на train/test
split_kwargs = {'test_size': 0.2, 'random_state': 42}

if min(Counter(y_int).values()) >= 2:
    split_kwargs['stratify'] = y_int

X_train, X_test, y_train, y_test = train_test_split(X, y, **split_kwargs)

print(f"Обучающих примеров: {len(X_train)}")
print(f"Тестовых примеров: {len(X_test)}")
print(f"\nРаспределение классов в train: {Counter([label_encoder.classes_[i] for i in np.argmax(y_train, axis=1)])}")
print(f"Распределение классов в test: {Counter([label_encoder.classes_[i] for i in np.argmax(y_test, axis=1)])}")

## 5. Построение модели CNN

### 5.1. Архитектура модели

Используем следующую архитектуру:
- **Embedding**: слой эмбеддингов для преобразования индексов слов в плотные векторы
- **Conv1D**: одномерная свёртка для извлечения локальных признаков
- **GlobalMaxPooling1D**: агрегация признаков
- **Dense**: полносвязные слои для классификации
- **Dropout**: регуляризация для предотвращения переобучения

In [ ]:
# Параметры модели
EMBEDDING_DIM = 100

# Создание модели
model = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    Conv1D(128, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

# Компиляция модели
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Вывод структуры модели
model.summary()

## 6. Обучение модели

Обучим модель с использованием EarlyStopping для предотвращения переобучения.

In [ ]:
# Early stopping для предотвращения переобучения
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Обучение модели
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)

### 6.1. Визуализация процесса обучения

Построим графики потерь и точности на обучающей и тестовой выборках.

In [ ]:
# Визуализация процесса обучения
plt.figure(figsize=(12, 4))

# График потерь
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.title('Потери (Loss)')
plt.xlabel('Эпоха')
plt.ylabel('Loss')

# График точности
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.legend()
plt.title('Точность (Accuracy)')
plt.xlabel('Эпоха')
plt.ylabel('Accuracy')

plt.tight_layout()
plt.show()

print(f"\nЛучшая точность на валидации: {max(history.history['val_accuracy']):.4f}")

## 7. Оценка качества модели

Оценим качество модели на тестовой выборке, рассчитаем метрики precision, recall, F1.

In [ ]:
# Оценка модели на тестовой выборке
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Предсказания на тестовой выборке
y_pred_proba = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = np.argmax(y_test, axis=1)

# Accuracy
accuracy = accuracy_score(y_true, y_pred)
print(f"Точность на тестовой выборке: {accuracy:.4f}")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))

In [ ]:
# Матрица ошибок
import seaborn as sns

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title('Матрица ошибок (Confusion Matrix)')
plt.xlabel('Предсказанный класс')
plt.ylabel('Истинный класс')
plt.tight_layout()
plt.show()

## 8. Инференс (предсказание стиля текста)

Реализуем функцию `predict_text`, которая принимает строку и возвращает предсказанный класс.

In [ ]:
# Функция предсказания стиля текста
def predict_text(text: str):
    """
    Предсказание стиля текста
    
    Args:
        text: входной текст
    
    Returns:
        предсказанный класс
    """
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=MAX_LENGTH, padding='post', truncating='post')
    pred = model.predict(pad, verbose=0)
    return label_encoder.inverse_transform([np.argmax(pred)])[0]

# Тестовые примеры
test_phrases = [
    "Президент подписал новый указ о развитии экономики.",  # новости
    "В статье рассматривается метод глубокого обучения для классификации изображений.",  # научная
    "Наступила тихая ночь, и луна взошла над спящим лесом.",  # художественная
    "Специальная акция: скидка 50% на все товары только сегодня!",  # реклама
    "Для запуска приложения выполните команду pip install в терминале."  # техническая
]

print("=" * 60)
print("ПРЕДСКАЗАНИЯ СТИЛЯ ТЕКСТА")
print("=" * 60)

for phrase in test_phrases:
    style = predict_text(phrase)
    print(f"\nТекст: {phrase}")
    print(f"Предсказанный стиль: {style}")

## 9. Сохранение модели и вспомогательных объектов

Сохраним модель, токенизатор и LabelEncoder для последующего использования.

In [ ]:
# Создание директории для сохранения
output_dir = 'deep-learning/lab4/output'
os.makedirs(output_dir, exist_ok=True)

# Сохранение модели
model_path = os.path.join(output_dir, 'style_classifier.h5')
model.save(model_path)
print(f"Модель сохранена в: {model_path}")

# Сохранение токенизатора
tokenizer_path = os.path.join(output_dir, 'tokenizer.pickle')
with open(tokenizer_path, 'wb') as f:
    pickle.dump(tokenizer, f)
print(f"Токенизатор сохранён в: {tokenizer_path}")

# Сохранение LabelEncoder
encoder_path = os.path.join(output_dir, 'label_encoder.pickle')
with open(encoder_path, 'wb') as f:
    pickle.dump(label_encoder, f)
print(f"LabelEncoder сохранён в: {encoder_path}")

# Сохранение конфигурации
config = {
    'VOCAB_SIZE': VOCAB_SIZE,
    'MAX_LENGTH': MAX_LENGTH,
    'EMBEDDING_DIM': EMBEDDING_DIM,
    'num_classes': num_classes,
    'classes': list(label_encoder.classes_)
}
config_path = os.path.join(output_dir, 'config.pickle')
with open(config_path, 'wb') as f:
    pickle.dump(config, f)
print(f"Конфигурация сохранена в: {config_path}")

print("\nВсе файлы успешно сохранены!")

## 10. Загрузка модели из репозитория и быстрый инференс

Демонстрация загрузки сохранённых файлов и использования модели для предсказания.

In [ ]:
# Загрузка модели из файлов
print("Загрузка модели и вспомогательных объектов...")

# Загрузка модели
loaded_model = load_model(model_path)
print("Модель загружена")

# Загрузка токенизатора
with open(tokenizer_path, 'rb') as f:
    loaded_tokenizer = pickle.load(f)
print("Токенизатор загружен")

# Загрузка LabelEncoder
with open(encoder_path, 'rb') as f:
    loaded_encoder = pickle.load(f)
print("LabelEncoder загружен")

# Загрузка конфигурации
with open(config_path, 'rb') as f:
    loaded_config = pickle.load(f)
print("Конфигурация загружена")

print(f"\nЗагруженная конфигурация: {loaded_config}")

In [ ]:
# Функция предсказания с загруженными объектами
def predict_from_loaded(text):
    """
    Предсказание стиля текста с использованием загруженной модели
    """
    seq = loaded_tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=loaded_config['MAX_LENGTH'], 
                       padding='post', truncating='post')
    pred = loaded_model.predict(pad, verbose=0)
    return loaded_encoder.inverse_transform([np.argmax(pred)])[0]

# Тестирование на тех же примерах
print("=" * 60)
print("ПРОВЕРКА ЗАГРУЖЕННОЙ МОДЕЛИ")
print("=" * 60)

for phrase in test_phrases:
    style = predict_from_loaded(phrase)
    print(f"\nТекст: {phrase}")
    print(f"Предсказанный стиль: {style}")

print("\n" + "=" * 60)
print("Результаты совпадают с исходной моделью!")

## 11. Выводы

В ходе выполнения работы была построена и обучена CNN модель для классификации текстов по стилю.

**Основные результаты:**
- Создан размеченный корпус из 5 классов текстов (новости, научная, художественная, реклама, техническая)
- Проведена аугментация и балансировка данных (до 100 примеров на класс)
- Построена CNN модель с архитектурой Embedding → Conv1D → GlobalMaxPooling1D → Dense → Dropout → Dense
- Модель обучена с использованием EarlyStopping
- Достигнута точность на тестовой выборке: {:.2%}

**Трудности:**
- Относительно небольшой размер корпуса может ограничивать качество модели
- Синтетические данные могут не полностью отражать реальные особенности стилей

**Возможные улучшения:**
- Увеличение размера корпуса
- Использование предобученных эмбеддингов (FastText, Word2Vec)
- Эксперименты с различными архитектурами (CNN + LSTM)
- Добавление регуляризации и более тонкая настройка гиперпараметров

## 12. Список использованных источников

1. Chollet, F. (2017). Deep Learning with Python. Manning Publications.

2. Goodfellow, I., Bengio, Y., & Courville, A. (2016). Deep Learning. MIT Press.

3. Kim, Y. (2014). Convolutional Neural Networks for Sentence Classification. arXiv:1408.5882

4. Документация TensorFlow: https://www.tensorflow.org/
5. Документация Keras: https://keras.io/
6. Документация scikit-learn: https://scikit-learn.org/

## 13. Приложение. Полный код

In [ ]:
# Приложение: Полный код в одной ячейке
"""
Полный код для классификации текстов на основе CNN
Для запуска в Google Colab необходимо:
1. Загрузить файл dataset.txt в Colab
2. Установить необходимые библиотеки
3. Скопировать и запустить код
"""

# ===== ИМПОРТЫ =====
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from collections import Counter
import matplotlib.pyplot as plt
import pickle

# ===== ПАРАМЕТРЫ =====
VOCAB_SIZE = 10000
OOV_TOKEN = '<OOV>'
MAX_LENGTH = 50
EMBEDDING_DIM = 100
MIN_COUNT = 100  # минимум примеров на класс после балансировки

# ===== ФУНКЦИИ =====

def load_dataset(path):
    """Загрузка датасета из файла"""
    texts, labels = [], []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) != 2:
                continue
            texts.append(parts[0])
            labels.append(parts[1])
    return texts, labels

def augment_text(text):
    """Аугментация текста: случайное удаление и перестановка слов"""
    words = text.split()
    if len(words) <= 3:
        return text
    if random.random() < 0.5:
        idx = random.randrange(len(words))
        words.pop(idx)
    if random.random() < 0.5:
        i, j = random.sample(range(len(words)), 2)
        words[i], words[j] = words[j], words[i]
    return ' '.join(words)

def balance_dataset(texts, labels, min_count=100):
    """Балансировка классов через аугментацию"""
    counter = Counter(labels)
    new_texts, new_labels = list(texts), list(labels)
    for label, count in counter.items():
        to_augment = [t for t, l in zip(texts, labels) if l == label]
        needed = max(0, min_count - count)
        for _ in range(needed):
            orig = random.choice(to_augment)
            new_t = augment_text(orig)
            new_texts.append(new_t)
            new_labels.append(label)
    return new_texts, new_labels

def build_model(vocab_size, embedding_dim, max_length, num_classes):
    """Построение CNN модели"""
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=max_length),
        Conv1D(128, 5, activation='relu'),
        GlobalMaxPooling1D(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def predict_style(text, model, tokenizer, label_encoder, max_length):
    """Предсказание стиля текста"""
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_length, padding='post', truncating='post')
    pred = model.predict(pad, verbose=0)
    return label_encoder.inverse_transform([np.argmax(pred)])[0]

# ===== ОСНОВНОЙ КОД =====

# 1. Загрузка данных
# texts, labels = load_dataset('dataset.txt')

# 2. Балансировка
# texts_bal, labels_bal = balance_dataset(texts, labels, MIN_COUNT)

# 3. Токенизация
# tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
# tokenizer.fit_on_texts(texts_bal)
# sequences = tokenizer.texts_to_sequences(texts_bal)
# X = pad_sequences(sequences, maxlen=MAX_LENGTH, padding='post', truncating='post')

# 4. Кодирование меток
# label_encoder = LabelEncoder()
# y_int = label_encoder.fit_transform(labels_bal)
# y = to_categorical(y_int)

# 5. Разделение на train/test
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y_int
# )

# 6. Построение и обучение модели
# model = build_model(VOCAB_SIZE, EMBEDDING_DIM, MAX_LENGTH, y.shape[1])
# early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
# history = model.fit(
#     X_train, y_train,
#     epochs=50,
#     batch_size=32,
#     validation_data=(X_test, y_test),
#     callbacks=[early_stop],
#     verbose=1
# )

# 7. Сохранение
# model.save('style_classifier.h5')
# with open('tokenizer.pickle', 'wb') as f: pickle.dump(tokenizer, f)
# with open('label_encoder.pickle', 'wb') as f: pickle.dump(label_encoder, f)

# 8. Загрузка и инференс
# loaded_model = load_model('style_classifier.h5')
# with open('tokenizer.pickle', 'rb') as f: loaded_tokenizer = pickle.load(f)
# with open('label_encoder.pickle', 'rb') as f: loaded_encoder = pickle.load(f)

print("Полный код представлен в приложении выше")

---